# Mission EdStats : supprimer les indicateurs redondants (phase 3)

La phase 2 a produit le tableau final pays × indicateurs (`country_indicator_final.csv` dans `data/output/`). Certains de ces indicateurs mesurent quasiment la même chose : pour academy, ils apportent la même information et alourdissent l'analyse sans rien y ajouter.

**Objectif** : ne garder que les indicateurs faiblement corrélés entre eux, c'est-à-dire un coefficient de corrélation (corrélation) inférieur à 0,70 en valeur absolue.

Plan de ce notebook : charger les CSV réduits de `data/output/` (section 1), déterminer le seuil de valeurs manquantes (section 2), calculer les matrices de corrélation de Pearson puis de Spearman et les visualiser en heatmap (carte de chaleur) (section 3), repérer les paires trop corrélées et choisir l'indicateur à garder (section 4), écrire le tableau réduit dans `data/output/` (section 5).

Flux des données : `data/output/` (entrée, produit par la phase 2) → ce notebook → `data/output/` (livrable de la phase 3).

## 1. Charger les CSV réduits de la phase 2

Les entrées de cette analyse sont les CSV écrits par la phase 2 dans `data/output/` : `country_indicator_final.csv` (tableau final pays × indicateurs) et `EdStatsSeries.csv` (nom complet de chaque code d'indicateur).

### 1.1 Dossiers et lecture

`load_csv_output()` lit un CSV de `data/output/`, que le notebook soit lancé depuis la racine du projet ou depuis `notebooks/`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

candidates = [
    Path.cwd() / "data" / "output",         # from the project root
    Path.cwd() / ".." / "data" / "output",  # from notebooks/
]

OUTPUT_DIR = None
for candidate in candidates:
    if candidate.exists():
        OUTPUT_DIR = candidate.resolve()
        break
if OUTPUT_DIR is None:
    raise FileNotFoundError("data/output/ not found")

def load_csv_output(name, columns=None):
    """Read the CSV `name` from OUTPUT_DIR as is, optionally keeping only `columns`."""
    return pd.read_csv(OUTPUT_DIR / name, encoding="utf-8", usecols=columns)

OUTPUT_DIR

PosixPath('/Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/output')

### 1.2 Le tableau final pays × indicateurs

Chargé une seule fois dans `final_table` et réutilisé par toutes les cellules suivantes : aucune opération ne modifie ce DataFrame en place, on recharge le CSV si besoin. Aperçu des 3 premières lignes :

In [2]:
# load once, reused by every cell of this notebook
final_table = load_csv_output("country_indicator_final.csv")
final_table.head(3)

,Country Code,Country Name,IT.NET.USER.P2,SP.SEC.UTOT.IN,SP.TER.TOTL.IN,SE.SEC.ENRL.GC,SE.TER.ENRL,SE.SEC.ENRR.UP,SE.TER.ENRR,SE.TOT.ENRR,PRJ.MYS.0T19.MF,PRJ.MYS.15UP.MF,PRJ.MYS.1519.MF,NY.GDP.PCAP.CD,SE.SEC.ENRL.TC.ZS,SE.XPD.TOTL.GD.ZS
0,ABW,Aruba,60.292077,4.478191e+03,6.799110e+03,6.564050e+03,1879.205882,90.496135,28.226781,85.283390,2.418519,8.777778,8.259259,24357.132282,14.201038,5.872939
1,AFG,Afghanistan,4.332797,1.865566e+06,2.624559e+06,1.910425e+06,137132.434783,32.586328,4.899025,63.099617,NaN,NaN,NaN,489.193612,35.668039,3.342328
2,AGO,Angola,5.180553,1.415692e+06,2.006447e+06,4.112328e+05,139698.582090,17.902200,6.445411,63.723745,NaN,NaN,NaN,3483.189042,25.717081,3.100720


Taille du tableau et liste des colonnes indicateurs (toutes les colonnes sauf `Country Code` et `Country Name`) :

In [3]:
print(f"{final_table.shape[0]} rows, {final_table.shape[1]} columns")

indicator_columns = []
for col in final_table.columns:
    if col not in ("Country Code", "Country Name"):
        indicator_columns.append(col)

print(f"{len(indicator_columns)} indicateurs:")
for col in indicator_columns:
    print(f"  {col}")

214 rows, 16 columns
14 indicateurs:
  IT.NET.USER.P2
  SP.SEC.UTOT.IN
  SP.TER.TOTL.IN
  SE.SEC.ENRL.GC
  SE.TER.ENRL
  SE.SEC.ENRR.UP
  SE.TER.ENRR
  SE.TOT.ENRR
  PRJ.MYS.0T19.MF
  PRJ.MYS.15UP.MF
  PRJ.MYS.1519.MF
  NY.GDP.PCAP.CD
  SE.SEC.ENRL.TC.ZS
  SE.XPD.TOTL.GD.ZS


### 1.3 Le nom des indicateurs

`EdStatsSeries.csv` associe chaque code (ex. `IT.NET.USER.P2`) à son nom complet. Référentiel chargé une seule fois dans `series`, réduit aux seuls indicateurs retenus par la phase 2 :

In [4]:
# load once, reused later for indicator names
series = load_csv_output("EdStatsSeries.csv", columns=["Series Code", "Indicator Name"])
series.head(3)

,Series Code,Indicator Name
0,IT.NET.USER.P2,Internet users (per 100 people)
1,NY.GDP.PCAP.CD,GDP per capita (current US$)
2,PRJ.MYS.0T19.MF,Projection: Mean years of schooling. Age 0-19....


### 1.4 Les valeurs manquantes (point de vigilance)

Point de vigilance : la matrice de corrélation ignore automatiquement les pays où un indicateur est non renseigné, paire par paire, ce qui peut fausser le calcul. Comptage des valeurs manquantes par indicateur et au total, avant toute corrélation :

In [5]:
nb_countries = len(final_table)
total_missing = 0
for col in indicator_columns:
    missing = final_table[col].isna().sum()
    total_missing += missing
    print(f"{col}: {missing} manquantes sur {nb_countries} pays")

total_cells = nb_countries * len(indicator_columns)
print(f"total: {total_missing} valeurs manquantes sur {total_cells} ({total_missing / total_cells:.1%})")

IT.NET.USER.P2: 9 manquantes sur 214 pays
SP.SEC.UTOT.IN: 17 manquantes sur 214 pays
SP.TER.TOTL.IN: 16 manquantes sur 214 pays
SE.SEC.ENRL.GC: 12 manquantes sur 214 pays
SE.TER.ENRL: 21 manquantes sur 214 pays
SE.SEC.ENRR.UP: 23 manquantes sur 214 pays
SE.TER.ENRR: 30 manquantes sur 214 pays
SE.TOT.ENRR: 30 manquantes sur 214 pays
PRJ.MYS.0T19.MF: 48 manquantes sur 214 pays
PRJ.MYS.15UP.MF: 48 manquantes sur 214 pays
PRJ.MYS.1519.MF: 48 manquantes sur 214 pays
NY.GDP.PCAP.CD: 7 manquantes sur 214 pays
SE.SEC.ENRL.TC.ZS: 32 manquantes sur 214 pays
SE.XPD.TOTL.GD.ZS: 32 manquantes sur 214 pays
total: 373 valeurs manquantes sur 2996 (12.4%)


### Bilan de la section 1

Tableau final chargé une seule fois depuis `data/output/`, noms des indicateurs via `EdStatsSeries.csv`, valeurs manquantes comptées par indicateur.

**Décision retenue** : le seuil de valeurs manquantes est déterminé en section 2, avant toute corrélation.

## 2. Seuil de valeurs manquantes

À quoi sert le seuil : un indicateur très troué fausse la matrice de corrélation, car `corr()` ignore les pays non renseignés paire par paire. Le taux de manquantes par indicateur décide si l'indicateur est assez complet pour rester dans l'analyse.

### 2.1 Taux de manquantes par indicateur

Trié du plus troué au plus complet, avec le nom :

In [6]:
# code -> name lookup from the series reference
name_of = {}
for i in range(len(series)):
    name_of[series["Series Code"].iloc[i]] = series["Indicator Name"].iloc[i]

# missing rate per indicator, sorted from most to least incomplete
rows = []
for col in indicator_columns:
    rows.append({
        "indicateur": col,
        "nom": name_of.get(col, ""),
        "taux_manquantes": final_table[col].isna().mean(),
    })

missing_rates = pd.DataFrame(rows).sort_values("taux_manquantes", ascending=False)
missing_rates

,indicateur,nom,taux_manquantes
8,PRJ.MYS.0T19.MF,Projection: Mean years of schooling. Age 0-19....,0.224299
9,PRJ.MYS.15UP.MF,Projection: Mean years of schooling. Age 15+. ...,0.224299
10,PRJ.MYS.1519.MF,Projection: Mean years of schooling. Age 15-19...,0.224299
12,SE.SEC.ENRL.TC.ZS,Pupil-teacher ratio in secondary education (he...,0.149533
13,SE.XPD.TOTL.GD.ZS,Government expenditure on education as % of GD...,0.149533
6,SE.TER.ENRR,"Gross enrolment ratio, tertiary, both sexes (%)",0.140187
7,SE.TOT.ENRR,"Gross enrolment ratio, primary to tertiary, bo...",0.140187
5,SE.SEC.ENRR.UP,"Gross enrolment ratio, upper secondary, both s...",0.107477
4,SE.TER.ENRL,"Enrolment in tertiary education, all programme...",0.098131
1,SP.SEC.UTOT.IN,Population of the official age for upper secon...,0.079439


In [7]:
# basis for the threshold: max missing rate outside the projections, and the projections' own rates
prj_codes = []
non_prj_codes = []
for col in indicator_columns:
    if col.startswith("PRJ."):
        prj_codes.append(col)
    else:
        non_prj_codes.append(col)

max_rate = 0
max_col = ""
for col in non_prj_codes:
    rate = final_table[col].isna().mean()
    if rate > max_rate:
        max_rate = rate
        max_col = col
nb_missing_max = int(final_table[max_col].isna().sum())
print(f"taux max hors projections : {nb_missing_max} manquantes sur {len(final_table)} pays = {max_rate:.2%} ({max_col})")

for col in prj_codes:
    nb = int(final_table[col].isna().sum())
    print(f"projection {col} : {nb} manquantes sur {len(final_table)} = {final_table[col].isna().mean():.2%}")

taux max hors projections : 32 manquantes sur 214 pays = 14.95% (SE.SEC.ENRL.TC.ZS)
projection PRJ.MYS.0T19.MF : 48 manquantes sur 214 = 22.43%
projection PRJ.MYS.15UP.MF : 48 manquantes sur 214 = 22.43%
projection PRJ.MYS.1519.MF : 48 manquantes sur 214 = 22.43%


**Conclusion** : seuil retenu **14,9 %** (14,95 % au brut : 32 pays manquants sur 214 hors projections), jugé raisonnable : tous les indicateurs passent le seuil, on les garde tous.

Note : les trois projections `PRJ.*` ont exactement le même taux (22,43 %, 48 pays sur 214 chacune), une certaine constance dans les données : on les garde aussi.